# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NadaFouad461/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

I build a 5-feature vector derived strictly from the February 2026 decision window: total impressions, total clicks, total sessions, average search position, and active reporting days. All features are aggregated at the client_hash_id × content_hash_id grain.

In [24]:
%pip -q install duckdb huggingface_hub pandas numpy

import getpass
import duckdb
import pandas as pd
import numpy as np
from huggingface_hub import login, hf_hub_download, list_repo_files

# Authentication
HF_TOKEN = getpass.getpass("Enter Hugging Face Token: ")
login(token=HF_TOKEN)

REPO_ID = "FlyRank/internship-warehouse"

#  Download Data
all_files = list_repo_files(repo_id=REPO_ID, repo_type="dataset", token=HF_TOKEN)
feb_files = [f for f in all_files if "month=2026-02" in f and f.endswith(".parquet")]
mar_files = [f for f in all_files if "month=2026-03" in f and f.endswith(".parquet")]

local_feb_paths = [hf_hub_download(repo_id=REPO_ID, filename=f, repo_type="dataset", token=HF_TOKEN) for f in feb_files]
local_mar_paths = [hf_hub_download(repo_id=REPO_ID, filename=f, repo_type="dataset", token=HF_TOKEN) for f in mar_files]

#  Connection Setup
con = duckdb.connect()
FACT_FEB = f"read_parquet({local_feb_paths})"
FACT_MAR = f"read_parquet({local_mar_paths})"

print(" Connection & Data Ready!")

Enter Hugging Face Token: ··········
 Connection & Data Ready!


In [25]:
feature_frame = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    COUNT(DISTINCT report_date) AS active_days_feb,
    SUM(gsc_clicks) AS total_clicks_feb,
    SUM(gsc_impressions) AS total_impressions_feb,
    SUM(ga4_sessions) AS total_sessions_feb,
    AVG(gsc_avg_position) AS avg_position_feb
FROM {FACT_FEB}
WHERE gsc_data_available IS TRUE
GROUP BY 1, 2
""").df()

print("Feature frame shape:", feature_frame.shape)
display(feature_frame.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (153559, 7)


,client_hash_id,content_hash_id,active_days_feb,total_clicks_feb,total_impressions_feb,total_sessions_feb,avg_position_feb
0,client_e547b89c05043229,content_1eea820697c3b95a,28,0.0,299.0,0.0,12.946228
1,client_e547b89c05043229,content_9abd8b303f805847,28,6.0,733.0,6.0,6.495085
2,client_e547b89c05043229,content_5f58c55cbfee172a,28,0.0,514.0,1.0,10.490023
3,client_e547b89c05043229,content_6fe390ba3af1e456,28,3.0,2931.0,6.0,38.436254
4,client_e547b89c05043229,content_3ad5d2160242b9ca,28,2.0,970.0,3.0,9.710810


### Feature notes

total_impressions_feb: Measures total observed search visibility during February 2026 before the March outcome window.

total_clicks_feb: Measures total observed search clicks during February 2026 before the March outcome window.

total_sessions_feb: Measures total observed sessions during February 2026 before the March outcome window.

avg_position_feb: Measures average observed search position during February 2026.

active_days_feb: Measures the number of February reporting days represented for the client-content pair.

All five features are derived from February 2026 and therefore precede the March outcome window.

In [26]:
# Check missing values for the 5 features
null_summary = feature_frame.isnull().sum().to_frame(name='missing_count')
display(null_summary)

# Fill nulls safely if any exist
feature_frame['total_sessions_feb'] = feature_frame['total_sessions_feb'].fillna(0)

if feature_frame['avg_position_feb'].isnull().sum() > 0:
    mean_pos = feature_frame['avg_position_feb'].mean()
    feature_frame['avg_position_feb'] = feature_frame['avg_position_feb'].fillna(mean_pos)

assert feature_frame.isnull().sum().sum() == 0
print("Data hygiene check passed: Zero missing values.")

,missing_count
client_hash_id,0
content_hash_id,0
active_days_feb,0
total_clicks_feb,0
total_impressions_feb,0
total_sessions_feb,81138
avg_position_feb,0


Data hygiene check passed: Zero missing values.


## 3. The leakage hunt
I deliberately add the March outcome (march_total_clicks) as a feature. Because this value is derived from the future outcome window, it should not be available at the February decision moment. A correlation of 1.0 demonstrates deliberate leakage. I then remove it and keep the honest feature set.

In [27]:
#  Get March outcome target
march_label = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_clicks) AS march_total_clicks
FROM {FACT_MAR}
WHERE gsc_data_available IS TRUE
GROUP BY 1, 2
""").df()

#  Inject deliberate leakage
leaky_frame = feature_frame.merge(
    march_label,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

leaky_frame["leaky_feature"] = leaky_frame["march_total_clicks"]

print("Rows with deliberate leakage test:", len(leaky_frame))
print(
    "Correlation between leaky feature and March outcome:",
    leaky_frame["leaky_feature"].corr(leaky_frame["march_total_clicks"])
)

#  Drop the leaked feature
clean_feature_frame = leaky_frame.drop(
    columns=["leaky_feature", "march_total_clicks"]
)
print("\nLeaky features removed successfully.")
print("Clean frame shape:", clean_feature_frame.shape)

Rows with deliberate leakage test: 134238
Correlation between leaky feature and March outcome: 1.0

Leaky features removed successfully.
Clean frame shape: (134238, 7)


## 4. What I excluded and why
client_hash_id: Excluded because it is a pseudonymous client identifier, not a predictive feature.

content_hash_id: Excluded because it identifies the content item without describing its underlying metrics.

march_total_clicks: Excluded because it belongs to the future outcome window (March 2026) and causes label leakage.

gsc_data_available IS FALSE rows: Excluded to eliminate non-reporting noise.

In [28]:
# Confirm final clean dataset
print("Final clean feature columns:", list(clean_feature_frame.columns))
assert "leaky_feature" not in clean_feature_frame.columns
assert "march_total_clicks" not in clean_feature_frame.columns
print("Exclusion assertion passed: Target and leaky columns are absent.")

Final clean feature columns: ['client_hash_id', 'content_hash_id', 'active_days_feb', 'total_clicks_feb', 'total_impressions_feb', 'total_sessions_feb', 'avg_position_feb']
Exclusion assertion passed: Target and leaky columns are absent.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.